# Phân loại cảm xúc tiếng Việt (VLSP v5) — Xử lý, Huấn luyện & Đánh giá (Colab T4 GPU)

Notebook thực hiện quy trình từ **xử lý dữ liệu thuần túy → huấn luyện 3 pipeline → đánh giá đầy đủ & giải thích mô hình** trên **Google Colab T4 miễn phí** (`Runtime -> Change runtime type -> T4 GPU -> Run all`):
- **TF-IDF + LinearSVC:** Chạy trực tiếp trên 20.000 đặc trưng sparse `float32` (không nén PCA/SVD).
- **CBOW + CNN + BiLSTM:** `Conv1d(128, k=3)` + mask theo độ dài thật → `BiLSTM(64×2)` → `Dropout(0.5)`, tối đa 15 epoch, Early Stopping (`patience=4`).
- **PhoBERT-base-v2:** Fine-tune theo effective batch `32`, tối đa 6 epoch, Early Stopping (`patience=2`).

Mọi cấu hình tiền xử lý (`NFC + gộp khoảng trắng`, lọc trùng & giao Test trước khi chia `80/20`, ngưỡng `300/256`, vocab `10k`, giữ hoa/thường cho PhoBERT) kế thừa trực tiếp từ notebook EDA CPU và được cài đặt độc lập để chạy thẳng trên Web Colab.

# 1. Thiết lập môi trường & Nạp dữ liệu

In [ ]:
import subprocess, sys, shutil

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'kagglehub>=0.3,<2', 'pyvi==0.1.1', 'scikit-learn>=1.4,<2', 'pandas>=2,<4',
    'matplotlib>=3.8,<4', 'transformers>=4.52,<5', 'py-vncorenlp==0.1.4',
    'torch>=2.6,<3', 'gensim>=4.4,<5', 'datasets>=3,<5', 'accelerate>=1,<2', 'joblib'
])
if shutil.which('java') is None:
    subprocess.run(['apt-get', 'update', '-qq'], check=False)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'openjdk-17-jre-headless'], check=True)
print('Đã sẵn sàng thư viện.')

In [ ]:
import faulthandler
faulthandler.enable()
import gc, hashlib, json, math, os, platform, random, re, time, unicodedata, urllib.request, warnings
from collections import Counter
from datetime import datetime, timezone
from functools import lru_cache
from importlib.metadata import version
from itertools import groupby
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader, Dataset as TorchDataset
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, classification_report,
                             confusion_matrix, f1_score, matthews_corrcoef)
from sklearn.model_selection import StratifiedGroupKFold, StratifiedShuffleSplit
from sklearn.svm import LinearSVC
from threadpoolctl import threadpool_limits

warnings.filterwarnings('ignore', category=UserWarning)
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

SEED = 42
RUN_TRAINING = True
SMOKE_TEST = os.environ.get('NOTEBOOK_SMOKE', '0') == '1'
LABELS = [-1, 0, 1]
CLASS_NAMES = ['negative', 'neutral', 'positive']
DATA_DIR = None
AUGMENTATION_LINEAGE = None
WORD2VEC_FILE = None
DATASET_HANDLE = 'anhphmduy/vlsp-sentiment-dataset/versions/5'
OUTPUT_PARENT = Path('vlsp_oxford_results')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RUN_DIR = OUTPUT_PARENT / (('smoke-' if SMOKE_TEST else 'full-') + RUN_ID)
RUN_DIR.mkdir(parents=True, exist_ok=False)

W2V_MAX_LENGTH = 300
MAX_VOCAB_SIZE = 10000
NEURAL_EPOCHS = 1 if SMOKE_TEST else 15
PHOBERT_EPOCHS = 1 if SMOKE_TEST else 6
PHOBERT_MAX_LENGTH = 256
PHOBERT_MODEL = 'vinai/phobert-base-v2'
PHOBERT_BATCH_SIZE = 8
PHOBERT_ACCUMULATION = 4  # effective batch = 32
ARCHITECTURES = ['TF-IDF + LinearSVC', 'CBOW + CNN + BiLSTM', 'PhoBERT-base-v2']
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type != 'cuda':
    print('CẢNH BÁO: Không thấy GPU CUDA. Trên Web Colab hãy chọn Runtime -> Change runtime type -> T4 GPU.')

torch.set_num_threads(2)
native_thread_limit = threadpool_limits(limits=2)

def seed_everything():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def write_json(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def release_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

seed_everything()
write_json(RUN_DIR / 'environment.json', {
    'python': sys.version, 'device': str(DEVICE),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'packages': {name: version(name) for name in ['torch', 'transformers', 'pandas', 'numpy', 'scikit-learn', 'gensim']},
})
print('Thiết bị:', DEVICE, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU', '| SMOKE:', SMOKE_TEST, '| Thư mục:', RUN_DIR)

In [ ]:
FILES = {
    'Base': ('vlsp_sentiment_train.csv',),
    'Augmented': ('vlsp_sentiment_train_augmented_v2.csv', 'augmented_v2.csv'),
    'Test': ('vlsp_sentiment_test.csv',),
}

def locate_inputs(root):
    return {
        name: next((root / filename for filename in filenames if (root / filename).is_file()), None)
        for name, filenames in FILES.items()
    }

def resolve_inputs():
    if DATA_DIR is not None:
        root = Path(DATA_DIR).resolve()
    else:
        candidates = [
            Path('data'), Path('.'),
            Path('/kaggle/input/vlsp-sentiment-dataset'),
            Path('/kaggle/input/datasets/anhphmduy/vlsp-sentiment-dataset'),
        ]
        root = next((p for p in candidates if all(locate_inputs(p).values())), None)
        if root is None:
            import kagglehub
            root = Path(kagglehub.dataset_download(DATASET_HANDLE))
            root = next(
                (p.parent for p in sorted(root.rglob(FILES['Base'][0])) if all(locate_inputs(p.parent).values())),
                root,
            )
        root = root.resolve()
    paths = locate_inputs(root)
    missing = ['/'.join(FILES[name]) for name, path in paths.items() if path is None]
    if missing:
        raise FileNotFoundError(f'Thiếu dữ liệu VLSP tại {root}: {", ".join(missing)}')
    return root, paths

def read_vlsp(path, split):
    for delimiter in ('\t', ','):
        header = pd.read_csv(path, sep=delimiter, encoding='utf-8-sig', nrows=0)
        if {'Class', 'Data'}.issubset(header.columns.str.strip()):
            break
    else:
        raise ValueError(f'{path.name} cần hai cột Class và Data.')
    frame = pd.read_csv(path, sep=delimiter, encoding='utf-8-sig', keep_default_na=False, dtype=str)
    frame.columns = frame.columns.str.strip()
    frame = frame[['Class', 'Data']].copy()
    labels = pd.to_numeric(frame['Class'], errors='coerce')
    invalid = ~labels.isin(LABELS)
    if frame.empty or invalid.any():
        raise ValueError(f'{path.name}: {int(invalid.sum())} nhãn sai hoặc tập rỗng.')
    frame['label'] = labels.astype(int)
    frame = frame.rename(columns={'Data': 'text'})
    frame['row'] = np.arange(len(frame))
    frame['sample_id'] = split.lower() + '-' + frame['row'].astype(str)
    return frame[['sample_id', 'row', 'label', 'text']]

data_root, paths = resolve_inputs()
raw = {name: read_vlsp(path, name) for name, path in paths.items()}
display(pd.DataFrame({name: frame['label'].value_counts() for name, frame in raw.items()}).reindex(LABELS))

# 2. Chuẩn bị & Xử lý dữ liệu thuần túy

Áp dụng trực tiếp các quy tắc rút ra từ notebook EDA:
1. Chuẩn hóa tối thiểu (`NFC + gộp khoảng trắng`), giữ nguyên phủ định, teencode, emoticon và dấu câu.
2. Loại bỏ bản sao dư, khóa mâu thuẫn nhãn nội bộ và các câu trùng với tập `Test` trước khi chia `80/20 Stratified Validation`.
3. Tạo 3 biểu diễn đầu vào riêng biệt: **TF-IDF** (`20.000` đặc trưng sparse), **CBOW** (`PyVi`, vocab `10.000`, `max_length=300`) và **PhoBERT** (giữ hoa/thường, `RDRSegmenter`, `max_length=256`).

In [ ]:
PREPROCESSING_VERSION = 'vlsp-eda-v4-compact'

def normalize_text(text):
    return ' '.join(unicodedata.normalize('NFC', text).split())

def text_key(text):
    return normalize_text(text).casefold()

def audit_inputs(frames):
    prepared = {
        name: frame.assign(
            text_clean=frame['text'].map(normalize_text),
            raw_key=frame['text'].map(lambda t: ' '.join(unicodedata.normalize('NFC', t).casefold().split())),
        )
        for name, frame in frames.items()
    }
    for frame in prepared.values():
        frame['clean_key'] = frame['text_clean'].map(text_key)

    test_frame = prepared['Test']
    if test_frame['clean_key'].eq('').any():
        raise ValueError('Test có văn bản rỗng sau chuẩn hóa.')

    audit = []
    development = pd.concat([prepared['Base'], prepared['Augmented']])
    conflicting = set(development.groupby('clean_key')['label'].nunique().loc[lambda s: s > 1].index)
    cleaned = {'Test': test_frame.copy()}
    for name, frame in prepared.items():
        empty = frame['clean_key'].eq('')
        raw_overlap = frame['raw_key'].isin(test_frame['raw_key']) & ~empty
        overlap = frame['clean_key'].isin(test_frame['clean_key']) & ~empty
        conflict = frame['clean_key'].isin(conflicting) & ~empty & ~raw_overlap & ~overlap
        entry = {
            'split': name,
            'input_rows': len(frame),
            'raw_blank': int(frame['raw_key'].eq('').sum()),
            'empty_after_cleaning': int(empty.sum()),
            'raw_duplicate_extras': int(frame.loc[~empty, 'raw_key'].duplicated().sum()),
            'normalized_duplicate_extras': int(frame.loc[~empty, 'clean_key'].duplicated().sum()),
            'raw_overlap_with_test': None if name == 'Test' else int(raw_overlap.sum()),
            'normalized_overlap_with_test': None if name == 'Test' else int(overlap.sum()),
            'conflicting_dev_labels': None if name == 'Test' else int(conflict.sum()),
        }
        if name != 'Test':
            cleaned[name] = frame.loc[~(empty | raw_overlap | overlap | conflict)].copy()
        else:
            entry['test_conflicting_keys'] = int((frame.groupby('clean_key')['label'].nunique() > 1).sum())
        audit.append(entry)
    return cleaned, pd.DataFrame(audit).set_index('split')

assert normalize_text('Ko tốt :)) ❤️') == 'Ko tốt :)) ❤️'
assert normalize_text('  KHÔNG   tốt! ') == 'KHÔNG tốt!'
cleaned, audit = audit_inputs(raw)
POLICY = {
    'version': PREPROCESSING_VERSION,
    'common': ['NFC', 'collapse_whitespace', 'preserve_original_text', 'preserve_negation_numbers_symbols'],
    'semantic_replacements': False,
    'elongation_reduction': False,
    'tfidf': {
        'lowercase': True,
        'token_pattern': '(?u)(?::[-]?[)(d]|\\^\\^|t\\.t|\\b\\w+\\b)',
        'ngram_range': [1, 2],
        'max_features': 20000,
        'min_df': 1,
        'sublinear_tf': True,
    },
    'cbow': {'lowercase': True, 'segmenter': 'PyVi', 'vocabulary_limit': 10000, 'max_length': 300, 'pad': 0, 'unk': 1},
    'phobert': {'preserve_case': True, 'segmenter': 'RDRSegmenter', 'max_length': 256, 'effective_batch': 32},
    'eda_sources': {name: sha256_file(paths[name]) for name in ['Base', 'Augmented', 'Test']},
    'architectures': ARCHITECTURES,
}
write_json(RUN_DIR / 'preprocessing_policy.json', POLICY)

In [ ]:
def read_lineage(path, frames):
    mapping = pd.read_csv(path)
    columns = ['aug_row', 'base_row']
    if not set(columns).issubset(mapping):
        raise ValueError('Lineage cần hai cột aug_row và base_row.')
    numbers = mapping[columns].apply(pd.to_numeric, errors='coerce')
    for column, name in zip(columns, ('Augmented', 'Base')):
        if not numbers[column].isin(frames[name]['row']).all():
            raise ValueError(f'Lineage có chỉ số {column} không hợp lệ.')
    numbers = numbers.astype(int).set_index('aug_row', verify_integrity=True)
    if set(numbers.index) != set(frames['Augmented']['row']):
        raise ValueError('Lineage phải ánh xạ đúng một lần cho mọi dòng Augmented.')
    parents = frames['Augmented']['row'].map(numbers['base_row'])
    if not parents.map(frames['Base'].set_index('row')['label']).eq(frames['Augmented']['label']).all():
        raise ValueError('Lineage có dòng Augmented mang nhãn khác dòng Base cha.')
    return numbers['base_row']

def unique_texts(frame):
    return frame.drop_duplicates('clean_key').reset_index(drop=True)

def partition_frames(development, frames, lineage_path):
    base = development['Base']
    aug = development['Augmented']
    validation = None
    if lineage_path is not None:
        parent_rows = read_lineage(lineage_path, frames)
        base_groups = frames['Base'].set_index('row')['text'].map(normalize_text).map(text_key)
        folds = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
        _, val_idx = next(folds.split(base, base['label'], groups=base['clean_key']))
        validation = unique_texts(base.iloc[val_idx])
        val_keys = set(validation['clean_key'])
        val_raw_keys = set(validation['raw_key'])
        aug = aug.loc[aug['row'].map(parent_rows).isin(base['row'])]
        base = base.loc[~base['clean_key'].isin(val_keys)]
        parent_keys = aug['row'].map(parent_rows).map(base_groups)
        validation_descendants = aug.loc[parent_keys.isin(val_keys)]
        held_out_keys = val_keys | set(validation_descendants['clean_key'])
        held_out_raw = val_raw_keys | set(validation_descendants['raw_key'])
        aug = aug.loc[~(parent_keys.isin(val_keys) | aug['clean_key'].isin(held_out_keys) | aug['raw_key'].isin(held_out_raw))]
    return {'Base': unique_texts(base), 'Augmented': unique_texts(aug)}, validation

lineage_path = Path(AUGMENTATION_LINEAGE) if AUGMENTATION_LINEAGE is not None else None
if lineage_path is None and (data_root / 'augmentation_lineage.csv').is_file():
    lineage_path = data_root / 'augmentation_lineage.csv'

training, validation = partition_frames(cleaned, raw, lineage_path)
val_splits = None
if validation is None:
    val_splits = {}
    for name in ('Base', 'Augmented'):
        full_frame = unique_texts(training[name])
        splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
        tr_idx, va_idx = next(splitter.split(full_frame, full_frame['label']))
        training[name] = full_frame.iloc[tr_idx].reset_index(drop=True)
        val_splits[name] = full_frame.iloc[va_idx].reset_index(drop=True)
test = cleaned['Test'].reset_index(drop=True)

partitions = {**training, 'Test': test}
if validation is not None:
    partitions['Validation'] = validation
elif val_splits is not None:
    for name, vframe in val_splits.items():
        partitions[f'{name}_val'] = vframe

for name, frame in partitions.items():
    if set(frame['label']) != set(LABELS):
        raise ValueError(f'{name} cần đủ cả ba lớp sau chia tập.')
for name, frame in training.items():
    holdouts = (test,) + ((validation,) if validation is not None else (val_splits[name],))
    for holdout in holdouts:
        for key in ('raw_key', 'clean_key'):
            if set(frame[key]) & set(holdout[key]):
                raise ValueError(f'Phát hiện rò rỉ giữa train ({name}) và holdout theo {key}.')

audit['fit_or_test_rows'] = pd.Series({name: len(frame) for name, frame in {'Test': test, **training}.items()})
audit.to_csv(RUN_DIR / 'data_audit.csv')
display(audit)
display(pd.DataFrame({name: frame['label'].value_counts() for name, frame in partitions.items()}).reindex(LABELS).fillna(0).astype(int))

In [ ]:
from pyvi import ViTokenizer
import py_vncorenlp
from transformers import AutoTokenizer

if shutil.which('java') is None:
    raise RuntimeError('Cần Java cho VnCoreNLP.')
os.environ.setdefault('JAVA_HOME', str(Path(shutil.which('java')).resolve().parent.parent))
segmenter_dir = Path('vncorenlp').resolve()
segmenter_dir.mkdir(exist_ok=True)
working_dir = Path.cwd()
try:
    for relative in ['VnCoreNLP-1.2.jar', 'models/wordsegmenter/vi-vocab', 'models/wordsegmenter/wordsegmenter.rdr']:
        target = segmenter_dir / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        if not target.is_file() or target.stat().st_size == 0:
            for attempt in range(3):
                try:
                    request = urllib.request.Request('https://raw.githubusercontent.com/vncorenlp/VnCoreNLP/master/' + relative, headers={'User-Agent': 'VLSP-notebook'})
                    with urllib.request.urlopen(request, timeout=45) as response, target.with_suffix(target.suffix + '.part').open('wb') as output:
                        shutil.copyfileobj(response, output)
                    target.with_suffix(target.suffix + '.part').replace(target)
                    assert target.stat().st_size > 0
                    break
                except Exception:
                    if attempt == 2:
                        raise
                    time.sleep(2)
        if relative.endswith('.jar'):
            with target.open('rb') as stream:
                assert stream.read(2) == b'PK', 'VnCoreNLP JAR không hợp lệ'
    segmenter = py_vncorenlp.VnCoreNLP(annotators=['wseg'], save_dir=str(segmenter_dir))
finally:
    os.chdir(working_dir)

@lru_cache(maxsize=None)
def phobert_words(text):
    return ' '.join(segmenter.word_segment(text))

for frame in partitions.values():
    frame['text_tfidf'] = frame.text_clean.str.lower()
    frame['words'] = frame.text_clean.str.lower().map(ViTokenizer.tokenize)
    frame['phobert_words'] = frame.text_clean.map(phobert_words)

try:
    segmenter.server.terminate()
except Exception:
    pass
subprocess.run(['pkill', '-f', 'VnCoreNLP'], check=False)
del segmenter
phobert_words.cache_clear()
phobert_tokenizer = AutoTokenizer.from_pretrained(PHOBERT_MODEL, use_fast=False)

if SMOKE_TEST:
    def small(frame, count):
        return pd.concat([g.sample(min(count, len(g)), random_state=SEED) for _, g in frame.groupby('label')]).sort_values('sample_id').reset_index(drop=True)
    training = {name: small(frame, 8) for name, frame in training.items()}
    test = small(test, 2)
    validation = small(validation, 2) if validation is not None else None
    val_splits = {name: small(frame, 2) for name, frame in val_splits.items()} if val_splits is not None else None

partitions = {**training, 'Test': test}
if validation is not None:
    partitions['Validation'] = validation
elif val_splits is not None:
    for name, vframe in val_splits.items():
        partitions[f'{name}_val'] = vframe

manifest = pd.concat([frame.assign(split=name)[['split', 'sample_id', 'label', 'clean_key']] for name, frame in partitions.items()], ignore_index=True)
manifest.to_csv(RUN_DIR / 'split_manifest.csv', index=False)
fit_mode = 'grouped_validation' if validation is not None else 'stratified_validation_no_lineage'
run_metadata = {
    'run_id': RUN_ID, 'smoke_test': SMOKE_TEST, 'seed': SEED, 'dataset': DATASET_HANDLE,
    'fit_mode': fit_mode, 'labels': LABELS, 'architectures': ARCHITECTURES,
    'sample_counts': {name: len(frame) for name, frame in partitions.items()},
    'sources': {name: {'filename': path.name, 'sha256': sha256_file(path)} for name, path in paths.items()},
    'epochs': {'neural': NEURAL_EPOCHS, 'phobert': PHOBERT_EPOCHS},
    'timing_definition': 'Fit preprocessor + model initialization + training/validation; excludes shared Word2Vec loading and test inference.',
    'phobert_effective_batch': PHOBERT_BATCH_SIZE * PHOBERT_ACCUMULATION,
    'manifest_sha256': sha256_file(RUN_DIR / 'split_manifest.csv'),
    'preprocessing': POLICY,
}
if lineage_path is not None:
    run_metadata['lineage_sha256'] = sha256_file(lineage_path)
write_json(RUN_DIR / 'run.json', run_metadata)
print('Hoàn tất tiền xử lý 3 biểu diễn | Chế độ chia tập:', fit_mode, '| Mẫu:', run_metadata['sample_counts'])

# 3. Xây dựng & Huấn luyện 3 Pipeline

Định nghĩa và huấn luyện lần lượt 3 pipeline trên cả `Base` và `Augmented`:
1. `TF-IDF + LinearSVC` (trực tiếp trên ma trận thưa 20.000 chiều)
2. `CBOW + CNN + BiLSTM` (`Conv1d(128, k=3)` + `BiLSTM(64×2)` + `Dropout(0.5)`, tối đa 15 epoch, `patience=4`)
3. `PhoBERT-base-v2` (tối đa 6 epoch, `patience=2`, effective batch 32)

In [ ]:
def make_tfidf_vectorizer():
    return TfidfVectorizer(
        max_features=POLICY['tfidf']['max_features'],
        ngram_range=tuple(POLICY['tfidf']['ngram_range']),
        min_df=POLICY['tfidf']['min_df'],
        sublinear_tf=POLICY['tfidf']['sublinear_tf'],
        lowercase=False,
        token_pattern=POLICY['tfidf']['token_pattern'],
        dtype=np.float32,
    )

class WordSequences(TorchDataset):
    def __init__(self, frame, vocabulary):
        self.tokens, self.lengths = [], []
        self.labels = (frame.label.to_numpy() + 1).astype(np.int64)
        for text in frame.words:
            seq = [vocabulary.get(word, 1) for word in text.split()[:W2V_MAX_LENGTH]] or [1]
            self.lengths.append(len(seq))
            self.tokens.append(seq + [0] * (W2V_MAX_LENGTH - len(seq)))
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, index):
        return {'tokens': torch.tensor(self.tokens[index]), 'lengths': torch.tensor(self.lengths[index]), 'labels': torch.tensor(self.labels[index])}

class PhoBERTInputs(TorchDataset):
    def __init__(self, frame):
        self.encoded = phobert_tokenizer(frame.phobert_words.tolist(), truncation=True, max_length=PHOBERT_MAX_LENGTH)
        self.labels = (frame.label.to_numpy() + 1).astype(np.int64)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, index):
        return {**{key: values[index] for key, values in self.encoded.items()}, 'labels': int(self.labels[index])}

def independent_embedding(weights):
    return nn.Embedding.from_pretrained(weights.detach().clone(), freeze=False, padding_idx=0)

def packed_mean(sequence, lengths, recurrent):
    packed = pack_padded_sequence(sequence, lengths.cpu(), batch_first=True, enforce_sorted=False)
    packed_output, _ = recurrent(packed)
    output, _ = pad_packed_sequence(packed_output, batch_first=True)
    mask = torch.arange(output.shape[1], device=output.device)[None, :] < lengths[:, None]
    return (output * mask[:, :, None]).sum(1) / lengths[:, None]

class CBOWCNNBiLSTM(nn.Module):
    def __init__(self, weights):
        super().__init__()
        self.embedding = independent_embedding(weights)
        self.conv = nn.Conv1d(weights.shape[1], 128, kernel_size=3, padding=1)
        self.lstm = nn.LSTM(128, 64, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(128, 3)
    def forward(self, tokens, lengths):
        mask = torch.arange(tokens.shape[1], device=tokens.device)[None, :] < lengths[:, None]
        embedded = self.embedding(tokens) * mask[:, :, None]
        convolved = F.relu(self.conv(embedded.transpose(1, 2))).transpose(1, 2) * mask[:, :, None]
        return self.fc(self.dropout(packed_mean(convolved, lengths, self.lstm)))

WORD_MODELS = {'CBOW + CNN + BiLSTM': CBOWCNNBiLSTM}

def build_text_model(architecture, embeddings=None):
    if architecture == 'TF-IDF + LinearSVC':
        return LinearSVC(C=1.0, random_state=SEED, max_iter=2000)
    if architecture in WORD_MODELS:
        return WORD_MODELS[architecture](embeddings)
    if architecture == 'PhoBERT-base-v2':
        from transformers import AutoModelForSequenceClassification
        return AutoModelForSequenceClassification.from_pretrained(PHOBERT_MODEL, num_labels=3)
    raise ValueError(f'Kiến trúc không được hỗ trợ: {architecture}')

def build_vocabulary(frame):
    counts = Counter(word for text in frame.words for word in text.split())
    return {word: index for index, (word, _) in enumerate(counts.most_common(MAX_VOCAB_SIZE - 2), start=2)}

def make_embeddings(vocabulary, word_vectors):
    rng = np.random.default_rng(SEED)
    matrix = rng.normal(0, 0.5, (len(vocabulary) + 2, word_vectors.vector_size)).astype(np.float32)
    matrix[0] = 0
    for word, index in vocabulary.items():
        if word in word_vectors:
            matrix[index] = word_vectors[word]
    return torch.from_numpy(matrix)

def neural_logits(model, architecture, batch):
    if architecture == 'PhoBERT-base-v2':
        return model(**{key: value.to(DEVICE) for key, value in batch.items() if key != 'labels'}).logits
    return model(batch['tokens'].to(DEVICE), batch['lengths'].to(DEVICE))

def make_neural_dataset(architecture, frame, preprocessor=None):
    if architecture == 'PhoBERT-base-v2':
        return PhoBERTInputs(frame)
    return WordSequences(frame, preprocessor)

def make_text_loader(dataset, architecture, shuffle=False):
    from transformers import DataCollatorWithPadding
    batch_size = PHOBERT_BATCH_SIZE if architecture == 'PhoBERT-base-v2' else 64
    collator = DataCollatorWithPadding(phobert_tokenizer) if architecture == 'PhoBERT-base-v2' else None
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, collate_fn=collator,
                      generator=torch.Generator().manual_seed(SEED), num_workers=0)

@torch.inference_mode()
def evaluate_text_neural(model, architecture, loader):
    model.eval()
    loss_sum, count, probs, truth = 0.0, 0, [], []
    for batch in loader:
        labels = batch['labels'].to(DEVICE)
        logits = neural_logits(model, architecture, batch)
        loss_sum += F.cross_entropy(logits, labels, reduction='sum').item()
        count += len(labels)
        truth.extend(labels.cpu().tolist())
        probs.append(logits.softmax(1).cpu().numpy())
    probs = np.concatenate(probs)
    pred = probs.argmax(1)
    return {'loss': loss_sum / count, 'accuracy': float(accuracy_score(truth, pred)),
            'macro_f1': float(f1_score(truth, pred, labels=[0, 1, 2], average='macro', zero_division=0))}, probs

def verify_word_models():
    weights = torch.randn(8, 6)
    weights[0] = 0
    left, right = CBOWCNNBiLSTM(weights).eval(), CBOWCNNBiLSTM(weights).eval()
    assert left.embedding.weight.data_ptr() != right.embedding.weight.data_ptr()
    assert left.embedding.weight.data_ptr() != weights.data_ptr()
    before = right.embedding.weight.detach().clone()
    with torch.no_grad():
        left.embedding.weight[2].add_(1)
    torch.testing.assert_close(right.embedding.weight, before)
    lengths = torch.tensor([3, 5])
    x = torch.tensor([[2, 3, 4, 0, 0], [2, 3, 4, 5, 6]])
    with torch.no_grad():
        torch.testing.assert_close(left(x, lengths), left(F.pad(x, (0, 4)), lengths), atol=1e-6, rtol=1e-5)
    print('Đã kiểm tra embedding độc lập và bất biến theo padding cho CBOW + CNN + BiLSTM.')

verify_word_models()

In [ ]:
from transformers import AutoModelForSequenceClassification, get_cosine_schedule_with_warmup

experiments = {}

def experiment_directory(architecture, dataset):
    slug = re.sub(r'[^a-z0-9]+', '-', architecture.lower()).strip('-')
    directory = RUN_DIR / 'models' / slug / dataset.lower()
    directory.mkdir(parents=True, exist_ok=True)
    return directory

def train_text_neural(model, architecture, frame, preprocessor, directory, val_frame=None):
    model.to(DEVICE)
    epochs = PHOBERT_EPOCHS if architecture == 'PhoBERT-base-v2' else NEURAL_EPOCHS
    train_dataset = make_neural_dataset(architecture, frame, preprocessor)
    train_loader = make_text_loader(train_dataset, architecture, shuffle=True)
    train_evaluation = make_text_loader(train_dataset, architecture)
    val_loader = make_text_loader(make_neural_dataset(architecture, val_frame, preprocessor), architecture) if val_frame is not None else None
    is_phobert = architecture == 'PhoBERT-base-v2'
    optimizer = (torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01) if is_phobert
                 else torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5))
    accumulation = PHOBERT_ACCUMULATION if is_phobert else 1
    total_steps = epochs * math.ceil(len(train_loader) / accumulation)
    scheduler = get_cosine_schedule_with_warmup(optimizer, min(100, max(0, total_steps - 1)), total_steps) if is_phobert else None
    scaler = torch.amp.GradScaler('cuda', enabled=is_phobert and DEVICE.type == 'cuda')
    history, best_score, bad_epochs, selected_epoch = [], -1.0, 0, 0
    checkpoint = directory / 'best.pt'
    for epoch in range(epochs):
        model.train()
        total_loss, sample_count = 0.0, 0
        optimizer.zero_grad(set_to_none=True)
        for index, batch in enumerate(train_loader):
            labels = batch['labels'].to(DEVICE)
            start = (index // accumulation) * accumulation * train_loader.batch_size
            window_samples = min(accumulation * train_loader.batch_size, len(train_dataset) - start)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=scaler.is_enabled()):
                logits = neural_logits(model, architecture, batch)
                loss = F.cross_entropy(logits, labels)
            if not torch.isfinite(loss):
                raise RuntimeError(f'Loss không hữu hạn: {architecture}, epoch {epoch + 1}')
            scaler.scale(loss * len(labels) / window_samples).backward()
            if (index + 1) % accumulation == 0 or index + 1 == len(train_loader):
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                old_scale = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                if scheduler is not None and scaler.get_scale() >= old_scale:
                    scheduler.step()
                optimizer.zero_grad(set_to_none=True)
            total_loss += loss.item() * len(labels)
            sample_count += len(labels)
        train_values, _ = evaluate_text_neural(model, architecture, train_evaluation)
        row = {
            'epoch': epoch + 1, 'train_loss': total_loss / sample_count,
            'train_accuracy': train_values['accuracy'], 'train_macro_f1': train_values['macro_f1'],
            'train_eval_loss': train_values['loss'],
        }
        if val_loader is not None:
            val_values, _ = evaluate_text_neural(model, architecture, val_loader)
            row.update({f'val_{key}': value for key, value in val_values.items()})
            if val_values['macro_f1'] > best_score:
                best_score, bad_epochs, selected_epoch = val_values['macro_f1'], 0, epoch + 1
                torch.save(model.state_dict(), checkpoint)
            else:
                bad_epochs += 1
        history.append(row)
        pd.DataFrame(history).to_csv(directory / 'history.csv', index=False)
        print(f'{architecture} | epoch {epoch + 1}/{epochs} | loss={row["train_loss"]:.4f} | train F1={row["train_macro_f1"]:.4f}'
              + (f' | val F1={row["val_macro_f1"]:.4f}' if val_loader is not None else ''), flush=True)
        if val_loader is not None and bad_epochs >= (2 if is_phobert else 4):
            print('Early stopping theo validation macro-F1.', flush=True)
            break
    if val_loader is None:
        selected_epoch = len(history)
        torch.save(model.state_dict(), checkpoint)
    else:
        model.load_state_dict(torch.load(checkpoint, map_location=DEVICE, weights_only=True))
    if is_phobert:
        model.save_pretrained(directory / 'pretrained')
        phobert_tokenizer.save_pretrained(directory / 'pretrained')
        checkpoint.unlink()
    del optimizer, scheduler, scaler, train_loader, train_evaluation, val_loader
    return {'history': history, 'selected_epoch': selected_epoch,
            'selection': 'validation_macro_f1' if val_frame is not None else 'fixed_final_epoch'}

if RUN_TRAINING:
    plt.close('all')
    for var_name in ['raw', 'cleaned']:
        if var_name in globals():
            del globals()[var_name]
    release_memory()
    from gensim.models import KeyedVectors
    vector_path = Path(WORD2VEC_FILE) if WORD2VEC_FILE is not None else data_root / 'vi-model-CBOW.bin'
    if not vector_path.is_file():
        raise FileNotFoundError(f'Thiếu vector CBOW: {vector_path}')
    run_metadata['word2vec'] = {'file': vector_path.name, 'sha256': sha256_file(vector_path)}
    write_json(RUN_DIR / 'run.json', run_metadata)
    vectors = None
    for architecture in ARCHITECTURES:
        if architecture in WORD_MODELS and vectors is None:
            vectors = KeyedVectors.load_word2vec_format(str(vector_path), binary=True)
        if architecture == 'PhoBERT-base-v2':
            del vectors
            vectors = None
            release_memory()
        for dataset_name, frame in training.items():
            seed_everything()
            directory = experiment_directory(architecture, dataset_name)
            print(f'\nHuấn luyện {architecture} / {dataset_name}: {len(frame):,} mẫu', flush=True)
            started = time.perf_counter()
            preprocessor = None
            if architecture.startswith('TF-IDF'):
                preprocessor = make_tfidf_vectorizer()
                matrix = preprocessor.fit_transform(frame.text_tfidf)
                joblib.dump(preprocessor, directory / 'vectorizer.joblib')
            if architecture == 'TF-IDF + LinearSVC':
                model = build_text_model(architecture)
                model.fit(matrix, frame.label.to_numpy() + 1)
                joblib.dump(model, directory / 'classifier.joblib')
                summary = {'history': [], 'selected_epoch': None, 'selection': 'single_fit',
                           'parameters': int(model.coef_.size + model.intercept_.size)}
            else:
                if architecture in WORD_MODELS:
                    preprocessor = build_vocabulary(frame)
                    write_json(directory / 'vocabulary.json', preprocessor)
                    weights = make_embeddings(preprocessor, vectors)
                    model = build_text_model(architecture, weights)
                    del weights
                else:
                    model = build_text_model(architecture)
                parameter_count = sum(p.numel() for p in model.parameters())
                val_frame = validation if validation is not None else (val_splits[dataset_name] if val_splits is not None else None)
                summary = train_text_neural(model, architecture, frame, preprocessor, directory, val_frame=val_frame)
                summary['parameters'] = parameter_count
            summary.update(architecture=architecture, dataset=dataset_name, train_seconds=time.perf_counter() - started,
                           fit_mode=fit_mode, fit_rows=len(frame), directory=str(directory))
            probe_frame = frame.head(2)
            if architecture == 'TF-IDF + LinearSVC':
                probe = model.decision_function(preprocessor.transform(probe_frame.text_tfidf))
            else:
                probe_loader = make_text_loader(make_neural_dataset(architecture, probe_frame, preprocessor), architecture)
                _, probe = evaluate_text_neural(model, architecture, probe_loader)
                del probe_loader
            np.save(directory / 'reload_probe.npy', probe)
            write_json(directory / 'training_summary.json', summary)
            experiments[(architecture, dataset_name)] = summary
            del model, preprocessor
            release_memory()
    assert len(experiments) == 6

In [ ]:
if RUN_TRAINING:
    for (architecture, dataset_name), result in experiments.items():
        if not result['history']:
            print(architecture, dataset_name, ': fit một lần, không có đường học theo epoch.')
            continue
        table = pd.DataFrame(result['history'])
        fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), constrained_layout=True)
        for ax, metric in zip(axes, ('loss', 'accuracy', 'macro_f1')):
            ax.plot(table.epoch, table[f'train_{metric}'], marker='.', label='train')
            if f'val_{metric}' in table:
                ax.plot(table.epoch, table[f'val_{metric}'], marker='.', label='validation')
            ax.axvline(result['selected_epoch'], color='gray', linestyle='--', label='checkpoint chọn')
            ax.set(xlabel='Epoch', ylabel=metric)
            ax.legend()
        fig.suptitle(f'{architecture} · {dataset_name} · checkpoint epoch {result["selected_epoch"]}')
        fig.savefig(Path(result['directory']) / 'learning_curves.png', dpi=140, bbox_inches='tight')
        plt.show()
        plt.close(fig)
        print(f'Caption: {architecture}/{dataset_name}, checkpoint epoch {result["selected_epoch"]}; chọn theo validation macro-F1.')

# 4. Đánh giá đầy đủ, Phân tích lỗi, Giải thích & Kết luận

In [ ]:
def load_text_experiment(architecture, result):
    directory = Path(result['directory'])
    preprocessor = None
    if architecture.startswith('TF-IDF'):
        preprocessor = joblib.load(directory / 'vectorizer.joblib')
    if architecture == 'TF-IDF + LinearSVC':
        return joblib.load(directory / 'classifier.joblib'), preprocessor
    if architecture == 'PhoBERT-base-v2':
        model = AutoModelForSequenceClassification.from_pretrained(directory / 'pretrained')
    else:
        state = torch.load(directory / 'best.pt', map_location='cpu', weights_only=True)
        preprocessor = json.loads((directory / 'vocabulary.json').read_text())
        model = WORD_MODELS[architecture](state['embedding.weight'])
        model.load_state_dict(state)
    return model.to(DEVICE).eval(), preprocessor

def score_frame(model, architecture, frame, preprocessor):
    if architecture == 'TF-IDF + LinearSVC':
        scores = model.decision_function(preprocessor.transform(frame.text_tfidf))
        return model.classes_[scores.argmax(1)] - 1, scores, 'margin'
    loader = make_text_loader(make_neural_dataset(architecture, frame, preprocessor), architecture)
    _, scores = evaluate_text_neural(model, architecture, loader)
    return scores.argmax(1) - 1, scores, 'probability'

results, predictions, per_class_records = [], [], []
if RUN_TRAINING:
    expected = {(architecture, name) for architecture in ARCHITECTURES for name in ('Base', 'Augmented')}
    assert set(experiments) == expected
    fig, axes = plt.subplots(3, 2, figsize=(11, 12), constrained_layout=True)
    for index, ((architecture, dataset_name), training_result) in enumerate(experiments.items()):
        model, preprocessor = load_text_experiment(architecture, training_result)
        directory = Path(training_result['directory'])
        probe_frame = training[dataset_name].head(2)
        _, reloaded_probe, _ = score_frame(model, architecture, probe_frame, preprocessor)
        np.testing.assert_allclose(reloaded_probe, np.load(directory / 'reload_probe.npy'), atol=1e-5, rtol=1e-4)
        started = time.perf_counter()
        predicted, scores, score_type = score_frame(model, architecture, test, preprocessor)
        truth = test.label.to_numpy()
        report = classification_report(truth, predicted, labels=LABELS, target_names=CLASS_NAMES, output_dict=True, zero_division=0)
        matrix = confusion_matrix(truth, predicted, labels=LABELS)
        summary = {
            'architecture': architecture, 'dataset': dataset_name,
            'accuracy': float(accuracy_score(truth, predicted)),
            'balanced_accuracy': float(balanced_accuracy_score(truth, predicted)),
            'macro_precision': float(report['macro avg']['precision']),
            'macro_recall': float(report['macro avg']['recall']),
            'macro_f1': float(report['macro avg']['f1-score']),
            'weighted_f1': float(report['weighted avg']['f1-score']),
            'mcc': float(matthews_corrcoef(truth, predicted)),
            'polarity_reversal_rate': float(np.mean(predicted[truth != 0] == -truth[truth != 0])),
            'support': len(test), 'parameters': training_result['parameters'],
            'train_seconds': training_result['train_seconds'],
            'inference_seconds': time.perf_counter() - started,
        }
        assert int(matrix.sum()) == len(test)
        assert np.isclose(matrix.trace() / matrix.sum(), summary['accuracy'])
        records = test[['sample_id', 'text', 'text_clean', 'label']].copy().rename(columns={'label': 'true_label'})
        records.insert(0, 'architecture', architecture)
        records.insert(1, 'dataset', dataset_name)
        records['predicted_label'] = predicted
        records['score_type'] = score_type
        records['score'] = scores.max(1)
        for j, label in enumerate(LABELS):
            records[f'score_{label}'] = scores[:, j]
        records.to_csv(directory / 'test_predictions.csv', index=False)
        write_json(directory / 'metrics.json', summary)
        write_json(directory / 'classification_report.json', report)
        pd.DataFrame(matrix, index=LABELS, columns=LABELS).to_csv(directory / 'confusion_counts.csv')
        for label, name in zip(LABELS, CLASS_NAMES):
            per_class_records.append({'architecture': architecture, 'dataset': dataset_name, 'label': label, 'class_name': name, **report[name]})
        results.append(summary)
        predictions.append(records)
        ax = axes.flat[index]
        ax.imshow(matrix, cmap='Blues', vmin=0, vmax=int(test.label.value_counts().max()))
        ax.set(xticks=range(3), yticks=range(3), xticklabels=LABELS, yticklabels=LABELS,
               xlabel='Dự đoán', ylabel='Nhãn thật', title=f'{architecture}\n{dataset_name}')
        for row, column in np.ndindex(matrix.shape):
            ax.text(column, row, str(matrix[row, column]), ha='center', va='center',
                    color='white' if matrix[row, column] > matrix.max() * 0.55 else 'black')
        print(architecture, dataset_name, {key: round(summary[key], 4) for key in ['accuracy', 'macro_f1']}, flush=True)
        display(pd.DataFrame(report).T.round(4))
        del model, preprocessor
        release_memory()
    fig.savefig(RUN_DIR / 'confusion-matrices.png', dpi=140, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    prediction_table = pd.concat(predictions, ignore_index=True)
    prediction_table.to_csv(RUN_DIR / 'predictions.csv', index=False)
    pd.DataFrame(per_class_records).to_csv(RUN_DIR / 'per_class_metrics.csv', index=False)
    write_json(RUN_DIR / 'results.json', {'run': run_metadata, 'results': results})

In [ ]:
if RUN_TRAINING:
    summary = pd.DataFrame(results)
    summary.to_csv(RUN_DIR / 'summary.csv', index=False)
    display(summary.sort_values('macro_f1', ascending=False).round(4))
    comparison = summary.pivot(index='architecture', columns='dataset', values='macro_f1').reindex(ARCHITECTURES)
    comparison['delta_pp'] = 100 * (comparison.Augmented - comparison.Base)
    comparison.to_csv(RUN_DIR / 'augmentation_delta.csv')
    display(comparison.round(4))
    ax = comparison[['Base', 'Augmented']].plot.barh(figsize=(10, 5), xlim=(0, 1))
    ax.set(xlabel='Test macro-F1', ylabel='', title='So sánh 3 pipeline trên cùng 1.050 mẫu Test (Base vs Augmented)')
    for container in ax.containers:
        ax.bar_label(container, fmt='%.3f', padding=3, fontsize=8.5)
    ax.figure.tight_layout()
    ax.figure.savefig(RUN_DIR / 'model-comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(ax.figure)
    display(summary[['architecture', 'dataset', 'parameters', 'train_seconds', 'inference_seconds']].round(2))
    print('Caption: Δ macro-F1 (Augmented − Base), điểm phần trăm:')
    for architecture, row in comparison.iterrows():
        print(f'{architecture}: {row.delta_pp:+.2f}')

In [ ]:
if RUN_TRAINING:
    mistakes = prediction_table.loc[prediction_table.true_label != prediction_table.predicted_label].copy()
    mistakes.to_csv(RUN_DIR / 'errors.csv', index=False)
    error_pairs = mistakes.groupby(['architecture', 'dataset', 'true_label', 'predicted_label']).size().rename('count').reset_index()
    error_pairs.to_csv(RUN_DIR / 'error_pairs.csv', index=False)
    for architecture in ARCHITECTURES:
        selected = mistakes.loc[(mistakes.architecture == architecture) & (mistakes.dataset == 'Augmented')]
        print(f'\n{architecture} (Augmented): {len(selected)} / {len(test)} câu sai')
        if selected.empty:
            continue
        display(selected.groupby(['true_label', 'predicted_label']).size().rename('count').reset_index())
        display(selected.sort_values(['score', 'sample_id'], ascending=[False, True])[
            ['sample_id', 'text', 'true_label', 'predicted_label', 'score_type', 'score']].head(5))

In [ ]:
if RUN_TRAINING:
    explanation_dir = RUN_DIR / 'explanations'
    explanation_dir.mkdir(exist_ok=True)
    svc_result = experiments[('TF-IDF + LinearSVC', 'Augmented')]
    svc, vectorizer = load_text_experiment('TF-IDF + LinearSVC', svc_result)
    terms = vectorizer.get_feature_names_out()
    rows = []
    fig, axes = plt.subplots(1, 3, figsize=(14, 5), constrained_layout=True)
    for label, ax, weights in zip(LABELS, axes, svc.coef_):
        indices = np.argsort(weights)[-12:]
        ax.barh(terms[indices], weights[indices])
        ax.set(title=f'LinearSVC · lớp {label}', xlabel='Trọng số tuyến tính')
        rows.extend({'label': label, 'term': str(terms[i]), 'weight': float(weights[i])} for i in indices)
    fig.savefig(explanation_dir / 'svc-weights.png', dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    pd.DataFrame(rows).to_csv(explanation_dir / 'svc_weights.csv', index=False)

    sample = test.sort_values('sample_id').iloc[0]
    x = vectorizer.transform([sample.text_tfidf])
    scores = svc.decision_function(x)[0]
    target = int(scores.argmax())
    contribution = np.asarray(x.multiply(svc.coef_[target]).toarray()).ravel()
    assert np.isclose(contribution.sum() + svc.intercept_[target], scores[target])
    nonzero = np.flatnonzero(contribution)
    indices = nonzero[np.argsort(np.abs(contribution[nonzero]))[-12:]]
    display(pd.DataFrame({'term': terms[indices], 'contribution': contribution[indices]}).sort_values('contribution'))
    print('Câu LinearSVC:', sample.sample_id, sample.text, '| lớp dự đoán:', target - 1)
    del svc, vectorizer

    model, _ = load_text_experiment('PhoBERT-base-v2', experiments[('PhoBERT-base-v2', 'Augmented')])
    model.eval()
    def model_digest(m):
        digest = hashlib.sha256()
        for value in m.state_dict().values():
            digest.update(value.detach().cpu().contiguous().numpy().tobytes())
        return digest.hexdigest()
    before_digest = model_digest(model)

    @torch.inference_mode()
    def phobert_probabilities(texts):
        output = []
        for start in range(0, len(texts), PHOBERT_BATCH_SIZE):
            encoded = phobert_tokenizer(texts[start:start + PHOBERT_BATCH_SIZE], padding=True, truncation=True,
                                        max_length=PHOBERT_MAX_LENGTH, return_tensors='pt').to(DEVICE)
            output.append(model(**encoded).logits.softmax(1).cpu().numpy())
        return np.concatenate(output)

    candidates = prediction_table.loc[(prediction_table.architecture == 'PhoBERT-base-v2') &
                                     (prediction_table.dataset == 'Augmented')].copy()
    candidates = candidates.merge(test[['sample_id', 'phobert_words']], on='sample_id', validate='one_to_one')
    candidates = candidates[candidates.phobert_words.str.split().str.len().between(1, 32)]
    candidates = candidates[candidates.phobert_words.map(lambda t: len(phobert_tokenizer.encode(t)) <= PHOBERT_MAX_LENGTH)]
    chosen = pd.concat([candidates.loc[(candidates.true_label == candidates.predicted_label) == correct]
                        .sort_values('sample_id').head(2) for correct in [True, False]])
    effects = []
    for row in chosen.itertuples():
        words = row.phobert_words.split()
        variants = [row.phobert_words] + [' '.join(words[:i] + words[i + 1:]) for i in range(len(words))]
        probabilities = phobert_probabilities(variants)
        target = int(probabilities[0].argmax())
        assert target - 1 == row.predicted_label
        delta = probabilities[0, target] - probabilities[1:, target]
        fig, ax = plt.subplots(figsize=(9, max(3, len(words) * 0.22)))
        ax.barh(range(len(words)), delta, color=['#287764' if value >= 0 else '#a74d35' for value in delta])
        ax.set(yticks=range(len(words)), yticklabels=words, xlabel='P(lớp dự đoán ban đầu) − P(sau bỏ từ)',
               title=f'{row.sample_id}: thật {row.true_label}, dự đoán {row.predicted_label}')
        fig.tight_layout()
        fig.savefig(explanation_dir / f'phobert-{row.sample_id}.png', dpi=140, bbox_inches='tight')
        plt.show()
        plt.close(fig)
        effects.extend({'sample_id': row.sample_id, 'word_index': i, 'word': word,
                        'true_label': int(row.true_label), 'predicted_label': int(row.predicted_label),
                        'probability_delta': float(value)} for i, (word, value) in enumerate(zip(words, delta)))
        print('Câu gốc:', row.text)
    pd.DataFrame(effects, columns=['sample_id', 'word_index', 'word', 'true_label', 'predicted_label', 'probability_delta']).to_csv(explanation_dir / 'phobert_occlusion.csv', index=False)
    assert model_digest(model) == before_digest
    del model
    release_memory()

    # Kiểm chứng lại từ file predictions.csv đã xuất
    saved = pd.read_csv(RUN_DIR / 'predictions.csv')
    for result in results:
        rows = saved.loc[(saved.architecture == result['architecture']) & (saved.dataset == result['dataset'])]
        assert rows.sample_id.tolist() == test.sample_id.tolist()
        np.testing.assert_array_equal(rows.true_label.to_numpy(), test.label.to_numpy())
        assert np.isclose(accuracy_score(rows.true_label, rows.predicted_label), result['accuracy'])
        assert np.isclose(f1_score(rows.true_label, rows.predicted_label, labels=LABELS, average='macro'), result['macro_f1'])
    write_json(RUN_DIR / 'verification.json', {'status': 'passed', 'runs': len(results), 'smoke_test': SMOKE_TEST,
                                              'checkpoint_reload': 'passed', 'explanation_weights_unchanged': True})
    best = summary.sort_values(['macro_f1', 'architecture', 'dataset'], ascending=[False, True, True]).iloc[0]
    print('\nKẾT LUẬN', 'SMOKE — chỉ kiểm chứng kỹ thuật' if SMOKE_TEST else 'FULL — seed 42')
    print(f'Dẫn đầu lần chạy: {best.architecture}/{best.dataset}, accuracy={100 * best.accuracy:.2f}%, macro-F1={100 * best.macro_f1:.2f}%.')
    print('Protocol:', fit_mode, '| Test:', len(test), 'mẫu | Một seed; augmentation không bảo đảm luôn cải thiện.')
    print('Kết quả:', RUN_DIR.resolve())